# BS5 C-R/C-N: full-state 24k to total 60k
Seeds 551/552 only. Kaggle datasets are already extracted. See BS5_CURRICULUM_EXTEND_60K_KAGGLE.md. Missing full parents or incompatible runtime stops before rollout.


In [ ]:
import os, sys, json, pathlib, subprocess, hashlib
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
INPUT = pathlib.Path('/kaggle/input')
WORK = pathlib.Path('/kaggle/working')
sources = sorted({p.parents[2] for p in INPUT.rglob('run_bs5_curriculum_extend.py') if p.parent.name == 'training'})
assert len(sources) == 1, f'Expected one current source: {sources}'
SRC = sources[0]
os.chdir(SRC)
NAMES = ['C-R_seed551','C-R_seed552','C-N_seed551','C-N_seed552']
parents = [p.parent for p in INPUT.rglob('protocol.json')
           if all((p.parent/'runs'/n/'resume.pt').is_file() for n in NAMES)
           and (p.parent/'source_snapshot.zip').is_file()]
assert len(parents) == 1, f'Need one full parent24k with all four resume.pt; audit_light is insufficient: {parents}'
PARENT = parents[0]
def protocol_hash(p):
    return hashlib.sha256(json.dumps(json.loads(p.read_text()), sort_keys=True, separators=(',',':'), allow_nan=False).encode()).hexdigest()
parent_hash = protocol_hash(PARENT/'protocol.json')
evidence = [p.parent for p in INPUT.rglob('protocol.json') if protocol_hash(p) == parent_hash]
EVIDENCE = max(evidence, key=lambda d: sum(1 for n in NAMES for _ in (d/'runs'/n/'evaluations').rglob('*.json.gz')))
OUT = WORK/'bs5_curriculum_extend_60k'
RUNNER = SRC/'scripts/training/run_bs5_curriculum_extend.py'
def run_command(*args):
    subprocess.run([sys.executable,'-B',str(RUNNER),*map(str,args)],cwd=SRC,check=True)
print('cd:', SRC, '\nparent:', PARENT, '\nevidence:', EVIDENCE, '\noutput:', OUT)


In [ ]:
# Match the parent runtime before rollout. Install matching builds and restart if needed.
import torch, numpy as np
print(sys.version, torch.__version__, np.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.__version__.startswith('2.11.') and np.__version__ == '2.1.3' and torch.cuda.is_available(), 'Need parent Torch2.11 CUDA / NumPy2.1.3; see README.'
if not (OUT/'protocol.json').exists():
    assert not OUT.exists(), 'Partial prepare: preserve evidence and choose a new output.'
    run_command('preflight','--parent',PARENT,'--output',WORK/'bs5_extend_preflight')
    run_command('prepare-extension','--parent',PARENT,'--evidence',EVIDENCE,'--output',OUT)
else:
    print('Already prepared; keep frozen protocol/parents and resume.')


In [ ]:
# Four tails only. Re-run this cell to resume. Stop at TOTAL 60k.
run_command('run','--output',OUT,'--resume')


In [ ]:
# Idempotent export; copy archives and closing receipt to Kaggle output.
import shutil
from IPython.display import FileLink, display
run_command('export','--output',OUT)
receipts = list((OUT/'exports').glob('receipt_*.json'))
assert receipts, 'No durable export snapshot yet.'
receipt = max(receipts,key=lambda p:p.stat().st_mtime_ns)
info = json.loads(receipt.read_text())
for item in info['artifacts']:
    src = OUT/item['path']
    digest = hashlib.sha256()
    with src.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''):
            digest.update(chunk)
    assert digest.hexdigest() == item['sha256']
    target = WORK/src.name
    shutil.copyfile(src,target)
    display(FileLink(str(target)))
target = WORK/receipt.name
shutil.copyfile(receipt,target)
display(FileLink(str(target)))
print('Download review_small, audit_light, repro_full and receipt. Stop at 60k.')
